# DGIdb Dataset Demo

This notebook demonstrates how to explore DGIdb data from Jupyter using the agentic stack.

1. Service setup and preflight checks
2. Health and A2A discovery
3. Direct DGIdb HTTP queries
4. Basic charting from query results

## 0. Setup

In [ ]:
import os
import json
import socket
import importlib

REQUIRED_PACKAGES = {
    "httpx": "httpx",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}

missing_packages = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    raise RuntimeError(
        "Missing packages detected: "
        f"{missing_packages}. Install them, then rerun this cell."
    )

import httpx
import pandas as pd
import matplotlib.pyplot as plt

JUPYTER_NODE_URL = os.environ.get("JUPYTER_NODE_URL", "http://localhost:8002")
DGIDB_NODE_URL = os.environ.get("SIBLING_DGIDB_NODE_URL", "http://dgidb-node:8005")

print(f"Jupyter node: {JUPYTER_NODE_URL}")
print(f"DGIdb node  : {DGIDB_NODE_URL}")

In [ ]:
# Preflight network and service checks.
dgidb_host = DGIDB_NODE_URL.replace("http://", "").replace("https://", "").split(":")[0]

try:
    socket.getaddrinfo(dgidb_host, None)
    print(f"DGIdb DNS: OK ({dgidb_host})")
except Exception as exc:
    print(f"DGIdb DNS: FAILED ({exc})")

for path in ["/health", "/ready", "/.well-known/a2a"]:
    try:
        resp = httpx.get(f"{DGIDB_NODE_URL}{path}", timeout=10)
        print(f"DGIdb {path}: {resp.status_code}")
    except Exception as exc:
        print(f"DGIdb {path}: FAILED ({exc})")

## 1. Read agent discovery card

In [ ]:
card_resp = httpx.get(f"{DGIDB_NODE_URL}/.well-known/a2a", timeout=10)
card_resp.raise_for_status()
card = card_resp.json()
print(json.dumps(card, indent=2))

## 2. Query DGIdb summary and schema

The exact routes may evolve during implementation. This cell tries common summary paths and prints whichever is available.

In [ ]:
candidate_paths = [
    "/data/summary",
    "/dgidb/summary",
    "/data/tables",
    "/dgidb/tables",
]

results = []
for path in candidate_paths:
    url = f"{DGIDB_NODE_URL}{path}"
    try:
        resp = httpx.get(url, timeout=10)
        results.append({"path": path, "status": resp.status_code, "body": resp.text[:400]})
    except Exception as exc:
        results.append({"path": path, "status": "error", "body": str(exc)})

pd.DataFrame(results)

## 3. Example DGIdb search/query

Adjust this request once final API route contracts are merged.

In [ ]:
payload = {
    "query": "EGFR",
    "limit": 20
}

candidate_query_paths = ["/dgidb/search", "/data/query"]
response_rows = []

for path in candidate_query_paths:
    try:
        resp = httpx.post(f"{DGIDB_NODE_URL}{path}", json=payload, timeout=20)
        response_rows.append({
            "path": path,
            "status": resp.status_code,
            "body_preview": resp.text[:500],
        })
    except Exception as exc:
        response_rows.append({
            "path": path,
            "status": "error",
            "body_preview": str(exc),
        })

pd.DataFrame(response_rows)

In [ ]:
# Optional quick visualization if one result set includes count-like fields.
# Update field names after API contracts are finalized.
example = {"interactions": 12, "genes": 7, "drugs": 4}
labels = list(example.keys())
values = list(example.values())

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.title("DGIdb Example Counts")
plt.xlabel("Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()